## Modelo predictivo de coste de seguro

## Cargamos el CSV y las librerias

In [21]:
import pandas as pd
from sklearn.model_selection import train_test_split
import statsmodels.api as sm
from sklearn.preprocessing import StandardScaler

In [22]:
df= pd.read_csv("Data/df_con_probabilidad.csv")

In [23]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 101886 entries, 0 to 101885
Data columns (total 24 columns):
 #   Column                                 Non-Null Count   Dtype  
---  ------                                 --------------   -----  
 0   ID                                     101886 non-null  str    
 1   Edad                                   101886 non-null  int64  
 2   Ingresos                               101886 non-null  float64
 3   Monto_Inicial                          101886 non-null  int64  
 4   Scoring_Crediticio                     101886 non-null  int64  
 5   Meses_Empleo                           101886 non-null  float64
 6   Num_Creditos                           101886 non-null  int64  
 7   Ratio_Interes                          101886 non-null  float64
 8   Duracion                               101886 non-null  float64
 9   Ratio_Deuda_Ingresos                   101886 non-null  float64
 10  Estudios                               101886 non-null  int64  
 11

In [24]:
df= df.drop(columns="Impago")

Eliminamos la columna impago porque es una clasificacion poco realista, Esto es debido a que la clasificacion depende de un umbral que nosotros mismos hemos elegido a proposito para que trate de detectar el maximo numero de impagos posibles. Es por eso que vamos a trabajar unicamente con la probabilidad de impago que nuestro modelo nos ha devuelto

## Separamos en TRAIN TEST y PUBLICO

In [25]:
df_modelo = df.copy()

df_modelo = df_modelo.drop(columns="ID")


df_publico = df_modelo.sample(frac=0.10, random_state=42)
df_modelo = df_modelo.drop(df_publico.index)


target = "Prima"
X = df_modelo.drop(columns=target)
y = df_modelo[target]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

## Regresion simple

In [26]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_train_scaled = pd.DataFrame(X_train_scaled, columns=X_train.columns, index=X_train.index)
X_test_scaled = scaler.transform(X_test)
X_test_scaled = pd.DataFrame(X_test_scaled, columns=X_test.columns, index=X_test.index)

In [27]:
X_train_scaled = sm.add_constant(X_train_scaled, prepend=True)
modelo_lin_reg = sm.OLS(endog=y_train, exog=X_train_scaled)
modelo_lin_reg = modelo_lin_reg.fit()
print(modelo_lin_reg.summary())

                            OLS Regression Results                            
Dep. Variable:                  Prima   R-squared:                       0.677
Model:                            OLS   Adj. R-squared:                  0.677
Method:                 Least Squares   F-statistic:                     8102.
Date:                Wed, 28 Oct 2026   Prob (F-statistic):               0.00
Time:                        16:11:44   Log-Likelihood:            -4.7248e+05
No. Observations:               73357   AIC:                         9.450e+05
Df Residuals:                   73337   BIC:                         9.452e+05
Df Model:                          19                                         
Covariance Type:            nonrobust                                         
                                            coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------------------------------

C:\Users\guill\AppData\Local\Temp\ipykernel_24024\126548182.py:3: SingularMatrixWarning: The design matrix is rank-deficient. The model parameters are not uniquely determined.
  modelo_lin_reg = modelo_lin_reg.fit()


Vamos a quitar las variables con un p valor mas alto que 0,05 ya que si este es mas alto que 0,05 no podriamos rechazar la hipotesis nula y por lo tanto esas variables no serian estadisticamente significativas

In [28]:
variables_quitar = [
    "Meses_Empleo",
    "Num_Creditos",
    "Posesion_Hipoteca",
    "Personas_Cargo",
    "Fiador",
    "Tipo_Jornada_Laboral_Autónomo",
    "Tipo_Jornada_Laboral_Desempleado",
    "Tipo_Jornada_Laboral_Jornada completa",
    "Tipo_Jornada_Laboral_Tiempo parcial",
    "Estado_Civil_Casado",
    "Estado_Civil_Divorciado",
    "Estado_Civil_Soltero"
]